# Gross Code LPU — Tour de Gross Convention

**Reference:** arXiv:2506.03094 ("Tour de Gross"), Appendix A.3 (LPU graph) and A.4 (measurement protocol)
**Source file:** `gross_code_lpu_tdg.py`

This notebook walks through every layer of `gross_code_lpu_tdg.py` interactively:

1. Why a different polynomial convention from Bravyi
2. The Monomial abstraction and qubit layout
3. Logical operator polynomials p, q, r, s
4. The LPU auxiliary graph G — vertices, edges, cycles, bridge
5. Physical ancilla qubit assignment
6. Sanity checks
7. Building the X̄₁ and Z̄₁ measurement circuits
8. Sampling and detector statistics
9. Detector Error Model

> **Convention note.** This file uses the *TDG* (Tour de Gross) polynomial convention:
> A = 1 + y + x³y⁻¹, B = 1 + x + x⁻¹y⁻³. The companion file
> `gross_lpu_analysis_bravyi.py` uses the original Bravyi convention.
> The two conventions label the same code but assign different physical indices
> to the same logical operators.


In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import stim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
%matplotlib inline

import gross_code_lpu_tdg as tdg
from bb_code_sim import ErrorModel

print('gross_code_lpu_tdg loaded')
print(f'Total qubits in full extended circuit: {tdg.N_TOTAL_QUBITS}')


---
## 1 — Why a Different Convention?

The Bravyi convention defines A = x³ + y + y², B = y³ + x + x².

The TDG (LPU-paper) convention defines A = 1 + y + x³y⁻¹, B = 1 + x + x⁻¹y⁻³.

Both generate the same [[144,12,12]] gross code, just with the lattice sites relabeled.
The LPU paper's specific logical-operator polynomials **p, q, r, s** (Appendix A.1)
are stated in the TDG convention, so the LPU graph construction requires TDG labels.


In [ ]:
print('TDG polynomial exponents:')
print(f'  A_EXPS = {tdg.A_EXPS}  ->  A = 1 + y + x^3 y^-1')
print(f'  B_EXPS = {tdg.B_EXPS}  ->  B = 1 + x + x^-1 y^-3')
print()
print(f'H_X shape: {tdg.H_X.shape}  (rows=X-checks, cols=data qubits)')
print(f'H_Z shape: {tdg.H_Z.shape}')
print()
print('CSS condition H_X @ H_Z.T = 0 mod 2:',
      np.all((tdg.H_X @ tdg.H_Z.T) % 2 == 0))


---
## 2 — The Monomial Class and Qubit Layout

Every data qubit is labeled by a monomial x^i y^j on the L or R block of the
12×6 torus. The `Monomial` dataclass wraps (side, i, j) and provides:

| Method | What it does |
|--------|--------------|
| `.qubit_index()` | Flat index: L → i·6+j, R → 72+i·6+j |
| `.transpose()` | ZX-duality: (αL)ᵀ = αᵀR, (αR)ᵀ = αᵀL |
| `.shift(di, dj)` | Multiply by x^di · y^dj |

Full qubit layout (378 total):

| Range | Block | Size |
|-------|-------|------|
| 0–71 | Gross-code L data | 72 |
| 72–143 | Gross-code R data | 72 |
| 144–215 | Gross-code X-ancilla | 72 |
| 216–287 | Gross-code Z-ancilla | 72 |
| 288–334 | LPU edge qubits | 47 |
| 335–358 | LPU vertex check qubits (Bell pair = 2) | 24 |
| 359–377 | LPU cycle check qubits | 19 |


In [ ]:
from gross_code_lpu_tdg import Mon

examples = [
    Mon('L', 4, 0),   # x^4 L
    Mon('R', 3, 5),   # x^3 y^5 R
    Mon('L', 9, 1),   # x^9 y L
    Mon('R', 4, 0),   # x^4 R  (the identified Bell-pair vertex)
]
print(f'{"Monomial":<20} {"qubit_index":<14} {"transpose":<20} {"transpose.qubit_index"}')
print('-' * 74)
for m in examples:
    t = m.transpose()
    print(f'{str(m):<20} {m.qubit_index():<14} {str(t):<20} {t.qubit_index()}')


---
## 3 — Logical Operator Polynomials p, q, r, s

From Appendix A.1 eq. (pqrs_gross), the four logical operators are:

- **X̄₁ = X(p, q)**: X on L-qubits p, R-qubits q
- **X̄₇ = X(r, s)**: X on L-qubits r, R-qubits s
- **Z̄₁ = Z(xy·sᵀ, xy·rᵀ)**: ZX-dual of X̄₇ shifted by xy
- **Z̄₇ = Z(xy·qᵀ, xy·pᵀ)**: ZX-dual of X̄₁ shifted by xy

Required algebraic properties:
- {X̄₁, Z̄₁} = −1 (anticommute — conjugate pair)
- [X̄₁, Z̄₇] = 0 and [X̄₇, Z̄₁] = 0 (cross-commute)


In [ ]:
from gross_code_lpu_tdg import (P, Q, R_POLY, S_POLY,
                                  X1_L, X1_R, X7_L, X7_R,
                                  Z1_L, Z1_R, Z7_L, Z7_R,
                                  _symplectic_overlap, Mon)

def support_indices(l_mono, r_mono):
    return sorted(
        [Mon('L', i, j).qubit_index() for i, j in l_mono] +
        [Mon('R', i, j).qubit_index() for i, j in r_mono]
    )

print('Logical operator supports (TDG qubit indices):')
for name, lp, rp in [('X̄₁', X1_L, X1_R), ('X̄₇', X7_L, X7_R),
                      ('Z̄₁', Z1_L, Z1_R), ('Z̄₇', Z7_L, Z7_R)]:
    supp = support_indices(lp, rp)
    print(f'  {name}  weight={len(supp)}  {supp}')

print()
print('Commutation relations:')
pairs = [
    ('X̄₁', X1_L, X1_R, 'Z̄₁', Z1_L, Z1_R, 1),
    ('X̄₁', X1_L, X1_R, 'Z̄₇', Z7_L, Z7_R, 0),
    ('X̄₇', X7_L, X7_R, 'Z̄₁', Z1_L, Z1_R, 0),
    ('X̄₇', X7_L, X7_R, 'Z̄₇', Z7_L, Z7_R, 1),
]
for xn, xL, xR, zn, zL, zR, expected in pairs:
    overlap = _symplectic_overlap(xL, xR, zL, zR)
    ok = 'OK' if overlap == expected else 'FAIL'
    rel = 'anticommute' if expected == 1 else 'commute'
    print(f'  [{xn}, {zn}] overlap={overlap}  expect {expected} ({rel})  {ok}')


---
## 4 — The LPU Auxiliary Graph G

The LPU is described by a graph G = (V, E) with cycle basis U (Appendix A.3).
Each element maps to a physical quantum object:

| Graph element | Physical role |
|---------------|---------------|
| Vertex v ∈ V  | X-check qubit (Bell-pair check for the identified vertex) |
| Edge e ∈ E    | Data qubit (LPU edge qubit) |
| Cycle u ∈ U   | Z-check qubit |

G is built in three stages:
1. **G_l** (12 vertices, 18 edges): built from X̄₁ support
2. **G_r** (12 vertices, 18 edges): built from X̄₇ support
3. **Merge**: identify x⁴R (∈ G_l) ≡ x⁹yL (∈ G_r) as a Bell-pair vertex; add 11 bridge edges


In [ ]:
from gross_code_lpu_tdg import (V_ALL, V_L_RAW, V_R_RAW, E_ALL,
                                  E_L_DETAIL, E_R_DETAIL, BRIDGE_EDGES,
                                  U_ALL, U_L, U_R, U_B, U_KIND,
                                  IDENTIFIED_L_SIDE, IDENTIFIED_R_SIDE)

print('=== LPU Graph Counts ===')
print(f'|V| = {len(V_ALL)}   (V_l=12 + V_r=12 - 1 identified)')
print(f'  identified: {IDENTIFIED_L_SIDE} (in V_l) = {IDENTIFIED_R_SIDE} (in V_r)')
print(f'|E| = {len(E_ALL)}   (E_l={len(E_L_DETAIL)} + E_r={len(E_R_DETAIL)} + bridge={len(BRIDGE_EDGES)})')
print(f'|U| = {len(U_ALL)}   (U_l={len(U_L)} + U_r={len(U_R)} + U_B={len(U_B)})')
n_vertex_qs = len(V_ALL) + 1  # +1 for Bell pair second qubit
print(f'Total ancilla = {len(E_ALL)} edges + {n_vertex_qs} vertex-checks (incl Bell) + {len(U_ALL)} cycle-checks = {len(E_ALL)+n_vertex_qs+len(U_ALL)}')


In [ ]:
# Vertex lists
print('V_l vertices (support of X̄₁ = pL + qR):')
for v in V_L_RAW:
    side, i, j = v
    qi = Mon(side, i, j).qubit_index()
    bell = '  <-- Bell-pair vertex' if v == IDENTIFIED_L_SIDE else ''
    print(f'  x^{i}y^{j}{side}  q={qi}{bell}')

print()
print('V_r vertices (support of X̄₇ = rL + sR):')
for v in V_R_RAW:
    side, i, j = v
    qi = Mon(side, i, j).qubit_index()
    bell = '  <-- Bell-pair vertex (identified)' if v == IDENTIFIED_R_SIDE else ''
    print(f'  x^{i}y^{j}{side}  q={qi}{bell}')


In [ ]:
# All 47 edges
from gross_code_lpu_tdg import EDGE_QUBIT, _frozen_edge_key

print(f'{"#":<4} {"gamma":<18} {"delta":<18} {"check":<8} {"kind":<6} {"edge_q"}')
print('-' * 64)
for k, (a, b, check_idx, kind) in enumerate(E_ALL):
    eq = EDGE_QUBIT[_frozen_edge_key((a, b, check_idx, kind))]
    cs = str(check_idx) if check_idx is not None else 'bridge'
    print(f'{k:<4} x^{a[1]}y^{a[2]}{a[0]:<12} x^{b[1]}y^{b[2]}{b[0]:<12} {cs:<8} {kind:<6} {eq}')


In [ ]:
# Cycle bases
from gross_code_lpu_tdg import CYCLE_EDGES, CYCLE_QUBIT

print('Cycle basis U (19 cycles):')
for k, (cycle, kind) in enumerate(zip(U_ALL, U_KIND)):
    verts = ' -> '.join(f'x^{v[1]}y^{v[2]}{v[0]}' for v in cycle)
    print(f'  U[{k:2d}] ({kind})  len={len(cycle)-1}  cycle_q={CYCLE_QUBIT[k]}')
    print(f'         {verts}')
    print(f'         edge_qs={CYCLE_EDGES[k]}')


### 4b — LPU Graph Visualization

The full auxiliary graph G drawn with NetworkX.
Blue = E_l / V_l (left half), orange = E_r / V_r (right half),
red = bridge edges, gold = Bell-pair vertex.


In [ ]:
G = nx.Graph()

def vname(v):
    return f'x^{v[1]}y^{v[2]}{v[0]}'

for v in V_ALL:
    G.add_node(vname(v))

edge_colors_list = []
for (a, b, check_idx, kind) in E_ALL:
    G.add_edge(vname(a), vname(b))
    edge_colors_list.append({'L': '#4C72B0', 'R': '#DD8452', 'B': '#C44E52'}[kind])

bell_node = vname(IDENTIFIED_L_SIDE)
node_colors_list = [
    'gold' if vname(v) == bell_node else
    ('#4C72B0' if v in V_L_RAW else '#DD8452')
    for v in V_ALL
]

fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, seed=42, k=1.8)
nx.draw_networkx(G, pos=pos, ax=ax,
                 node_color=node_colors_list, node_size=500, font_size=7,
                 edge_color=edge_colors_list, width=2.0)

legend_handles = [
    mpatches.Patch(color='#4C72B0', label='V_l / E_l  (left half-LPU, measures X̄₁ / Z̄₇)'),
    mpatches.Patch(color='#DD8452', label='V_r / E_r  (right half-LPU, measures X̄₇ / Z̄₁)'),
    mpatches.Patch(color='#C44E52', label='Bridge edges (expand fault distance)'),
    mpatches.Patch(color='gold',    label='Bell-pair vertex  (x^4 R = x^9 y L)'),
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=9)
ax.set_title('LPU Auxiliary Graph G  (23 vertices, 47 edges)', fontsize=13)
plt.tight_layout()
plt.show()


---
## 5 — Physical Qubit Assignment

Each graph element is assigned a concrete Stim qubit index:

- **Edge qubits** 288–334: one per edge, in `E_ALL` order
- **Vertex check qubits** 335–358: one per vertex, except the Bell-pair vertex which occupies two consecutive indices (one per half-LPU)
- **Cycle check qubits** 359–377: one per cycle in `U_L + U_R + U_B` order


In [ ]:
from gross_code_lpu_tdg import (VERTEX_QUBIT, EDGE_QUBIT_BASE,
                                  VERTEX_QUBIT_BASE, CYCLE_QUBIT_BASE,
                                  N_TOTAL_QUBITS)

print('Qubit index layout:')
print(f'  [0,   72)   Gross-code L data (TDG convention)')
print(f'  [72, 144)   Gross-code R data')
print(f'  [144, 216)  Gross-code X-ancilla')
print(f'  [216, 288)  Gross-code Z-ancilla')
print(f'  [{EDGE_QUBIT_BASE}, {VERTEX_QUBIT_BASE})  LPU edge qubits        ({VERTEX_QUBIT_BASE-EDGE_QUBIT_BASE} qubits)')
print(f'  [{VERTEX_QUBIT_BASE}, {CYCLE_QUBIT_BASE})  LPU vertex-check qubits ({CYCLE_QUBIT_BASE-VERTEX_QUBIT_BASE} qubits; Bell=2)')
print(f'  [{CYCLE_QUBIT_BASE}, {N_TOTAL_QUBITS})  LPU cycle-check qubits  ({N_TOTAL_QUBITS-CYCLE_QUBIT_BASE} qubits)')
print(f'  Total: {N_TOTAL_QUBITS}')
print()
print('Bell-pair vertex physical qubits:')
print(f'  (id, "l") -> q={VERTEX_QUBIT[(IDENTIFIED_L_SIDE, "l")]}  (used in X̄₁ branch)')
print(f'  (id, "r") -> q={VERTEX_QUBIT[(IDENTIFIED_L_SIDE, "r")]}  (used in Z̄₁ branch)')


---
## 6 — Sanity Checks

The module runs `_sanity_check()` at import time. Here we reproduce each
assertion explicitly so you can see exactly what is being verified.


In [ ]:
from gross_code_lpu_tdg import (_check_commutes_all, _symplectic_overlap,
                                  H_X, H_Z)

checks = {
    '|V| = 23':          len(V_ALL) == 23,
    '|E| = 47':          len(E_ALL) == 47,
    '|E_l| = 18':        len(E_L_DETAIL) == 18,
    '|E_r| = 18':        len(E_R_DETAIL) == 18,
    '|bridge| = 11':     len(BRIDGE_EDGES) == 11,
    '|U| = 19':          len(U_ALL) == 19,
    'ancilla = 90':      (len(E_ALL) + len(V_ALL) + 1 + len(U_ALL)) == 90,
    'H_X shape (72,144)': H_X.shape == (72, 144),
    'CSS H_X@H_Z.T=0':   np.all((H_X @ H_Z.T) % 2 == 0),
    'X̄₁ in ker(H_Z)':    _check_commutes_all(X1_L, X1_R, H_Z),
    'X̄₇ in ker(H_Z)':    _check_commutes_all(X7_L, X7_R, H_Z),
    'Z̄₁ in ker(H_X)':    _check_commutes_all(Z1_L, Z1_R, H_X),
    'Z̄₇ in ker(H_X)':    _check_commutes_all(Z7_L, Z7_R, H_X),
    '{X̄₁,Z̄₁} anticommute': _symplectic_overlap(X1_L, X1_R, Z1_L, Z1_R) == 1,
    '[X̄₁,Z̄₇] commute':    _symplectic_overlap(X1_L, X1_R, Z7_L, Z7_R) == 0,
    '[X̄₇,Z̄₁] commute':    _symplectic_overlap(X7_L, X7_R, Z1_L, Z1_R) == 0,
}

all_pass = True
for name, result in checks.items():
    icon = 'OK' if result else 'FAIL'
    all_pass &= result
    print(f'  [{icon}] {name}')

print()
print('All checks passed.' if all_pass else 'SOME CHECKS FAILED.')


---
## 7 — Building the X̄₁ and Z̄₁ Measurement Circuits

The gauging-measurement protocol (Appendix A.4) runs in four stages:

1. **Initialize** LPU edge qubits (|0> for X̄₁ branch; |+> for Z̄₁ branch)
2. **Deformed-code rounds** (C=10): measure all LPU checks; some gross-code checks
   are extended to also act on edge qubits
3. **Return**: measure all edge qubits in Z; resume bare syndrome
4. **Logical result** = XOR of all vertex-check outcomes from the final LPU round

Each branch uses a *half-LPU*:

| Branch | Active subgraph | Data prep |
|--------|-----------------|-----------|
| X̄₁ | E_l + V_l + U_l + v_Bell (half-LPU l) | Data in \|+⟩ |
| Z̄₁ | E_r + V_r + U_r + v_Bell (half-LPU r) | Data in \|0⟩ |

Circuit macro-structure:
```
[d_init=12 bare BB rounds]  [C=10 deformed-code LPU rounds]  [edge Z measurements]  [d_init=12 bare BB rounds]  [final data measurement]
```


In [ ]:
from gross_code_lpu_tdg import build_logical_x1_circuit, build_logical_z1_circuit

em0 = ErrorModel(p_phys=0.0, p_meas=0.0)

print('Building X̄₁ circuit (p=0) ...')
circ_x1 = build_logical_x1_circuit(em0, C=10, d_init=12)
print(f'  qubits       = {circ_x1.num_qubits}')
print(f'  measurements = {circ_x1.num_measurements}')
print(f'  detectors    = {circ_x1.num_detectors}')
print(f'  observables  = {circ_x1.num_observables}')

print()
print('Building Z̄₁ circuit (p=0) ...')
circ_z1 = build_logical_z1_circuit(em0, C=10, d_init=12)
print(f'  qubits       = {circ_z1.num_qubits}')
print(f'  measurements = {circ_z1.num_measurements}')
print(f'  detectors    = {circ_z1.num_detectors}')
print(f'  observables  = {circ_z1.num_observables}')


In [ ]:
# Preview first and last few Stim instructions for X̄₁
lines = str(circ_x1).split('\n')
print(f'Total Stim instruction lines: {len(lines)}')
print()
print('--- First 30 lines ---')
for l in lines[:30]:
    print(' ', l)
print()
print('--- Last 12 lines ---')
for l in lines[-12:]:
    print(' ', l)


---
## 8 — Sampling: Noiseless Verification

At p=0, every detector must fire 0 times and the observable must be constant
across all shots. This is the basic correctness check for the circuit.


In [ ]:
SHOTS = 200

for name, circ in [('X̄₁', circ_x1), ('Z̄₁', circ_z1)]:
    sampler = circ.compile_detector_sampler(seed=0)
    dets, obs = sampler.sample(SHOTS, separate_observables=True)
    print(f'{name} at p=0  ({SHOTS} shots):')
    print(f'  total detector firings = {int(dets.sum())}  (should be 0)')
    print(f'  observable flips       = {int(obs.sum())}  (should be 0)')
    print()


### 8b — Noisy Sampling and Detector Histogram


In [ ]:
P_PHYS = 1e-3
em_noisy = ErrorModel(p_phys=P_PHYS, p_meas=P_PHYS)

print(f'Building noisy circuits at p={P_PHYS} ...')
circ_x1_noisy = build_logical_x1_circuit(em_noisy, C=10, d_init=12)
circ_z1_noisy = build_logical_z1_circuit(em_noisy, C=10, d_init=12)
print('Done.')


In [ ]:
for name, circ in [('X̄₁', circ_x1_noisy), ('Z̄₁', circ_z1_noisy)]:
    sampler = circ.compile_detector_sampler(seed=1)
    dets, obs = sampler.sample(SHOTS, separate_observables=True)
    mean_fired = dets.sum(axis=1).mean()
    n_obs_flip = int(obs.sum())
    print(f'{name} at p={P_PHYS}  ({SHOTS} shots):')
    print(f'  mean detectors fired/shot = {mean_fired:.1f} / {circ.num_detectors}')
    print(f'  raw observable flips      = {n_obs_flip} ({100*n_obs_flip/SHOTS:.1f}%)')
    print()


In [ ]:
# Histogram for X̄₁
sampler = circ_x1_noisy.compile_detector_sampler(seed=2)
dets, _ = sampler.sample(500, separate_observables=True)
fps = dets.sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(fps, bins=30, color='steelblue', edgecolor='white')
ax.set_xlabel('Detector firings per shot')
ax.set_ylabel('Count')
ax.set_title(f'X̄₁ LPU circuit — detector firings at p={P_PHYS}  (500 shots)')
plt.tight_layout()
plt.show()
print(f'Mean: {fps.mean():.1f},  Std: {fps.std():.1f}')


---
## 9 — Detector Error Model (DEM)

Stim converts the noisy circuit into a *detector error model* — a hypergraph
of error mechanisms, each linking the detectors it flips and observables it
corrupts. This is the input a decoder (e.g. BP+OSD) operates on.


In [ ]:
for name, circ in [('X̄₁', circ_x1_noisy), ('Z̄₁', circ_z1_noisy)]:
    dem = circ.detector_error_model(
        decompose_errors=False,
        ignore_decomposition_failures=True,
    )
    n_err = sum(1 for inst in dem.flattened() if inst.type == 'error')
    print(f'{name} DEM:')
    print(f'  detectors        = {dem.num_detectors}')
    print(f'  observables      = {dem.num_observables}')
    print(f'  error mechanisms = {n_err}')
    print()


---
## Summary

### What `gross_code_lpu_tdg.py` provides

| Symbol / function | What it is |
|-------------------|------------|
| `A_EXPS`, `B_EXPS` | TDG polynomial convention exponents |
| `H_X`, `H_Z` | 72×144 parity-check matrices (TDG) |
| `P`, `Q`, `R_POLY`, `S_POLY` | Logical-operator polynomials from Appendix A.1 |
| `X1_L/R`, `Z1_L/R`, `X7_L/R`, `Z7_L/R` | Qubit supports of the four logical operators |
| `V_L_RAW`, `V_R_RAW` | Vertex lists for G_l and G_r |
| `E_L_DETAIL`, `E_R_DETAIL` | Edges with associated gross-code check index |
| `BRIDGE_EDGES` | 11 bridge edges connecting G_l to G_r |
| `U_L`, `U_R`, `U_B` | Cycle bases (5 + 3 + 11 = 19 cycles) |
| `EDGE_QUBIT`, `VERTEX_QUBIT`, `CYCLE_QUBIT` | Physical qubit index lookups |
| `build_logical_x1_circuit(em, C, d_init)` | Full X̄₁ measurement circuit |
| `build_logical_z1_circuit(em, C, d_init)` | Full Z̄₁ measurement circuit |

### TDG vs Bravyi at a glance

| | `gross_lpu_analysis_bravyi.py` | `gross_code_lpu_tdg.py` |
|--|--|--|
| Convention | Bravyi | TDG / LPU-paper |
| A polynomial | x³+y+y² | 1+y+x³y⁻¹ |
| Purpose | Code structure, Tanner graph, visualization | LPU graph + Stim circuit generation |
| Qubit indices | Different | Different |
